In [30]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import numpy as np
def load_signal(file):
    df = pd.read_csv(
        file,
        sep=";",
        skiprows=7,
        names=["timestamp", "value"]
    )

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="%d.%m.%Y %H:%M:%S,%f"
    )

    return df
nasal = load_signal("/content/Flow - 30-05-2024.txt")
thoracic = load_signal("/content/Thorac - 30-05-2024.txt")
spo2 = load_signal("/content/SPO2 - 30-05-2024.txt")

#load event file
events = pd.read_csv(
    "/content/Flow Events - 30-05-2024.txt",
    sep=";",
    skiprows=5,
    names=["time_range", "duration", "event", "stage"]
)
#split the start and end time
events[["start", "end"]] = events["time_range"].str.split("-", expand=True)
#add date to the end timestamp
events["date"] = events["start"].str[:10]
events["end"] = events["date"] + " " + events["end"]
#convert to datetime
events["start"] = pd.to_datetime(
    events["start"],
    format="%d.%m.%Y %H:%M:%S,%f"
)
events["end"] = pd.to_datetime(
    events["end"],
    format="%d.%m.%Y %H:%M:%S,%f"
)

#filter
def bandpass_filter(signal, lowcut, highcut, fs, order=4):

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order, [low, high], btype='band')

    filtered = filtfilt(b, a, signal)

    return filtered

#apply the filter
fs = 32

nasal["filtered"] = bandpass_filter(
    nasal["value"],
    0.17,
    0.4,
    fs
)
thoracic["filtered"] = bandpass_filter(
    thoracic["value"],
    0.17,
    0.4,
    fs
)

#30 sec window with 50% overlap

def create_windows(values, timestamps, fs, window_sec=30, overlap=0.5):

    window_size = int(fs * window_sec)
    step = int(window_size * (1 - overlap))

    windows = []

    for start in range(0, len(values) - window_size + 1, step):

        end = start + window_size

        window_values = values[start:end]

        start_time = timestamps[start]
        end_time = timestamps[end-1]

        windows.append((start_time, end_time, window_values))

    return windows

nasal_windows = create_windows(nasal["filtered"].values,nasal["timestamp"],32)
thoracic_windows = create_windows(thoracic["filtered"].values,thoracic["timestamp"],32)
spo2_windows = create_windows(spo2["value"].values,spo2["timestamp"],4)
def label_windows(windows, events, window_sec=30):

    labels = []
    threshold = window_sec * 0.5   # 15 seconds

    for window in windows:

        window_start = window[0]
        window_end = window[1]

        label = "Normal"

        for _, event in events.iterrows():

            event_start = event["start"]
            event_end = event["end"]

            overlap_start = max(window_start, event_start)
            overlap_end = min(window_end, event_end)

            overlap = (overlap_end - overlap_start).total_seconds()

            if overlap > threshold:
                label = event["event"]
                break

        labels.append(label)

    return labels
labeled_windows=label_windows(nasal_windows, events)

In [45]:
dataset = []

for i in range(len(nasal_windows)):

    dataset.append({
        "start": nasal_windows[i][0],
        "end": nasal_windows[i][1],
        "nasal": nasal_windows[i][2],
        "thoracic": thoracic_windows[i][2],
        "spo2": spo2_windows[i][2],
        "label": labeled_windows[i]
    })

In [49]:
dataset_df = pd.DataFrame(dataset)
dataset_df.head()

,start,end,nasal,thoracic,spo2,label
0,2024-05-30 20:59:00,2024-05-30 20:59:29.969,"[35.79589475300924, 34.51643935193731, 33.0870...","[7.090939814271049, 7.28229806505017, 7.454058...","[93, 94, 94, 94, 94, 94, 94, 94, 94, 94, 94, 9...",Normal
1,2024-05-30 20:59:15,2024-05-30 20:59:44.969,"[101.79798806330602, 106.20436874110517, 110.3...","[3.3070819237407028, 4.119591930008364, 4.9209...","[93, 93, 93, 93, 93, 94, 94, 94, 94, 93, 93, 9...",Normal
2,2024-05-30 20:59:30,2024-05-30 20:59:59.969,"[-69.52902468045042, -75.03650915370152, -80.2...","[-1.0672884119494013, -1.7746894291620938, -2....","[93, 93, 93, 93, 93, 93, 93, 93, 93, 93, 93, 9...",Normal
3,2024-05-30 20:59:45,2024-05-30 21:00:14.969,"[34.44904372310536, 26.72274611345321, 18.8999...","[-21.761831624331016, -22.11488036038428, -22....","[93, 93, 93, 93, 93, 93, 93, 93, 93, 93, 93, 9...",Normal
4,2024-05-30 21:00:00,2024-05-30 21:00:29.969,"[-75.75679943140895, -82.00682020205466, -87.9...","[-6.461603429631263, -8.774200177925167, -11.0...","[93, 93, 93, 93, 93, 93, 93, 93, 93, 93, 93, 9...",Normal


In [51]:
dataset_df.to_csv("dataset_df.csv", index=False)

In [50]:
dataset_df.to_pickle("dataset_df.pkl")